# 实现测试Cute DSL copy kernel

> sync->async(实际用)->TMA(对比了解)

## 1D copy

### 1. 测试共用

In [54]:
"""tests/test_m3a_copy.py — 3a 三版 copy kernel 完整自包含实现 + 验收。

关键修复（相对上一版）：
  ① kernel 写成类方法（带 self），不写成模块级函数——避免 PHASE_DYNAMIC_TO_STATIC_BOOL
  ② 越界处理用 cute.where（staged SelectOp），不用裸 if
  ③ 循环用 cutlass.range（官方推荐；cutlass.range_dynamic 在 4.1.0 已废弃【changelog】）
  ④ to_cute 的 leading_dim 按 3D 张量 (B,N,d) 为 2，并统一走 invoke()
  ⑤ async 版按 issue #2980 的官方 predication 模式实现【ref: cutlass issue 2980】

跑法：
  uv run python tests/test_m3a_copy.py
"""
import time

import pytest
import torch

import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack

DK = 64
HAS_CUDA = torch.cuda.is_available()
CAP = torch.cuda.get_device_capability(0) if HAS_CUDA else (0, 0)
SM = int(f"{CAP[0]}{CAP[1]}")


# ================================================================ torch → CuTe（统一入口）

def to_cute(t: torch.Tensor):
    """3D (B, N, d) 连续张量：stride=1 的 mode 是 d（index 2），所以 leading_dim=2。"""
    assert t.is_contiguous(), "copy kernel 要求连续张量（slicing 后请先 .contiguous()）"
    return from_dlpack(t, assumed_align=16).mark_layout_dynamic(leading_dim=2)


def invoke(impl, Q_t: torch.Tensor, Qo_t: torch.Tensor):
    """唯一合法的调用入口：所有测试与 runner 都走这里，杜绝漏转换。"""
    impl(to_cute(Q_t), to_cute(Qo_t))

### 2. 同步(sync)版

In [55]:
# ================================================================ 版本 1：Sync（方法形式 + 裸 if）

class CopySync:
    """gmem → 寄存器 → gmem。每线程按 flat 下标算 (b, n, d) 坐标，裸 if 做 OOB 保护。"""

    def __init__(self, num_threads: int = 256):
        self.num_threads = num_threads

    @cute.kernel
    def kernel(self, mSrc: cute.Tensor, mDst: cute.Tensor,
               n_nd: cutlass.Int32, d: cutlass.Int32, total: cutlass.Int32):
        tidx, _, _ = cute.arch.thread_idx()
        bidx, _, _ = cute.arch.block_idx()
        bdim, _, _ = cute.arch.block_dim()

        idx = bidx * bdim + tidx
        if idx < total:                       # 方法形式 kernel 内的裸 if：官方模式
            b   = idx // n_nd
            rem = idx - b * n_nd
            n   = rem // d
            dd  = rem - n * d
            mDst[b, n, dd] = mSrc[b, n, dd]   # 3D 元组索引

    @cute.jit
    def __call__(self, mSrc: cute.Tensor, mDst: cute.Tensor):
        total = cute.size(mSrc)               # B*N*d，动态 Int32
        d = mSrc.shape[2]
        n_nd = mSrc.shape[1] * d
        gx = (total + self.num_threads - 1) // self.num_threads
        self.kernel(mSrc, mDst, n_nd, d, total).launch(
            grid=(gx, 1, 1), block=(self.num_threads, 1, 1)
        )


### 3. 异步(async)版

In [56]:
"""3a 异步版 copy 的修正：只改了 pred 的形状（其余沿用你的版本）。

用法：把下面的 CopyAsync 整个替换到 tests/test_m3a_copy.py 里的同名类即可；其余测试代码不变。
我没有在 DSL 上运行过它，以你的 4.8.1 的实际报错为准。
"""
import cutlass
import cutlass.cute as cute


class CopyAsync:
    """gmem --cp.async--> smem --> gmem。仅支持 bf16（vec=8 = 16B 粒度）。"""

    def __init__(self, num_threads: int = 256):
        self.num_threads = num_threads

    @cute.kernel
    def kernel(self, flat_src: cute.Tensor, flat_dst: cute.Tensor,
               tiled_copy: cute.TiledCopy,
               vec: cutlass.Constexpr, span: cutlass.Constexpr,
               total: cutlass.Int32):
        tidx, _, _ = cute.arch.thread_idx()
        bidx, _, _ = cute.arch.block_idx()

        smem = cutlass.utils.SmemAllocator()
        sBuf = smem.allocate_tensor(flat_src.element_type, cute.make_layout((span,)), 16)

        gBlock = cute.local_tile(flat_src, (span,), (bidx,))   # 本 block 的静态 (span,) 块

        thr_copy = tiled_copy.get_slice(tidx)
        tSgS = thr_copy.partition_S(gBlock)                    # 形状 ((8,1),((1)))：(每条指令的向量, 其余)
        tSsS = thr_copy.partition_D(sBuf)

        my_first = bidx * span + tidx * vec                    # 本线程 16B 块的全局首下标
        in_bounds = my_first < total                           # total 恒为 vec 的倍数 → 整块判断即可

        # ---- 修正点：pred 是"每个 16B 向量一个"，不是"每个元素一个" ----
        # 错误信息里的 expects (1,((1))) 就是这个意思：源张量的第 0 模 (8,1) 里，
        # 8 是一条 cp.async 指令搬的元素数（由 atom 的 128 位决定，不需要谓词），
        # 只有后面的 1（向量重复数）和其余模式才需要谓词。
        # 官方 SGEMM 例子的写法同理：(tAgA.shape[0][1], size(mode=[1]), size(mode=[2]))。
        n_rep = tSgS.shape[0][1]                               # 静态 int
        n_rest = cute.size(tSgS, mode=[1])                     # 静态 int
        pred = cute.make_rmem_tensor((n_rep, n_rest), cutlass.Boolean)
        for rv in range(n_rep):                                # 普通 Python range：编译期展开
            for r in range(n_rest):
                pred[rv, r] = in_bounds

        cute.copy(tiled_copy, tSgS, tSsS, pred=pred)           # 顶层调用，不包 if

        cute.arch.cp_async_commit_group()
        cute.arch.cp_async_wait_group(0)
        cute.arch.barrier()                                    # 所有线程都要执行，所以在 if 之外

        if in_bounds:
            for i in range(vec):                               # vec 是 Constexpr → 静态展开
                flat_dst[my_first + i] = sBuf[tidx * vec + i]

    @cute.jit
    def __call__(self, mSrc: cute.Tensor, mDst: cute.Tensor):
        assert mSrc.element_type == cutlass.BFloat16, "async 版 vec=8 按 bf16 写的"
        total = cute.size(mSrc)
        flat_src = cute.make_tensor(mSrc.iterator, cute.make_layout((total,)))
        flat_dst = cute.make_tensor(mDst.iterator, cute.make_layout((total,)))

        vec = 8
        span = self.num_threads * vec                          # 每 block 2048 元素
        atom = cute.make_copy_atom(cute.nvgpu.cpasync.CopyG2SOp(), mSrc.element_type, num_bits_per_copy=128)
        tiled_copy = cute.make_tiled_copy_tv(
            atom,
            cute.make_layout((self.num_threads,)),
            cute.make_layout((vec,)),
        )
        self.kernel(flat_src, flat_dst, tiled_copy, vec, span, total).launch(
            grid=((total + span - 1) // span, 1, 1),
            block=(self.num_threads, 1, 1),
            smem=span * 2,                                     # bf16 = 2 字节
        )

### TMA版

In [57]:

# ================================================================ 版本 3：TMA（sm80 干净拒绝）

class CopyTMA:
    """TMA 需要 sm90+。sm80 上必须显式拒绝，不允许静默跑错。"""

    def __call__(self, mSrc, mDst):
        if SM < 90:
            raise NotImplementedError(f"TMA 需要 sm90+，当前 sm{SM}")
        raise NotImplementedError("sm90 机器上对照官方 TMA 例子补实现（接口不变）")



### 测试

In [58]:

# ================================================================ 测试

IMPLS = {"sync": CopySync, "async": CopyAsync}
IMPL_DTYPES = {"sync": [torch.bfloat16, torch.float32], "async": [torch.bfloat16]}
SHAPES = [(1, 64), (2, 100), (1, 129)]


@HAS_CUDA and pytest.mark.skipif(not HAS_CUDA, reason="需要 GPU") or (lambda f: f)
def _noop():
    pass


needs_cuda = pytest.mark.skipif(not HAS_CUDA, reason="需要 GPU")


@needs_cuda
@pytest.mark.parametrize("dtype", [torch.bfloat16, torch.float32])
@pytest.mark.parametrize("batch,N", SHAPES)
def test_sync_identity(batch, N, dtype):
    torch.manual_seed(0)
    Q = torch.randn(batch, N, DK, device="cuda", dtype=dtype)
    Q_out = torch.zeros_like(Q)
    invoke(CopySync(), Q, Q_out)
    torch.cuda.synchronize()
    assert torch.equal(Q_out, Q), f"sync B={batch} N={N} {dtype} 不一致"


@needs_cuda
@pytest.mark.parametrize("trial", range(5))
def test_async_identity(trial):
    """跑 5 遍抓竞态：漏 wait/barrier 的 bug 是概率性的。"""
    torch.manual_seed(trial)
    Q = torch.randn(2, 100, DK, device="cuda", dtype=torch.bfloat16)
    Q_out = torch.zeros_like(Q)
    invoke(CopyAsync(), Q, Q_out)
    torch.cuda.synchronize()
    assert torch.equal(Q_out, Q), f"async 第 {trial} 次不一致（竞态？查 wait/barrier）"


@needs_cuda
@pytest.mark.parametrize("impl", ["sync", "async"])
def test_copy_preserves_bit_patterns(impl):
    """NaN/±Inf/±0 的位模式必须原样——按 int16 位模式比。"""
    Q = torch.zeros(1, 64, DK, device="cuda", dtype=torch.bfloat16)
    Q[0, 0, 0] = float("nan")
    Q[0, 0, 1] = float("inf")
    Q[0, 0, 2] = float("-inf")
    Q[0, 0, 3] = float("-0.0")
    Q_out = torch.zeros_like(Q)
    invoke(IMPLS[impl](), Q, Q_out)
    torch.cuda.synchronize()
    assert torch.equal(Q_out.view(torch.int16), Q.view(torch.int16))


@needs_cuda
def test_tma_rejects_cleanly_on_sm80():
    try:
        CopyTMA()(to_cute(torch.randn(1, 64, DK, device="cuda", dtype=torch.bfloat16)),
                  to_cute(torch.zeros(1, 64, DK, device="cuda", dtype=torch.bfloat16)))
    except NotImplementedError:
        pytest.skip(f"sm{SM} 不支持 TMA，类正确拒绝")
    pytest.fail("TMA 未拒绝")


@needs_cuda
def test_dynamic_shape_same_binary():
    """layout dynamic：同 impl 换 N 不应触发重编译（两次都应秒级）。"""
    Q1 = torch.randn(1, 64, DK, device="cuda", dtype=torch.bfloat16)
    Q2 = torch.randn(2, 100, DK, device="cuda", dtype=torch.bfloat16)
    impl = CopySync()
    invoke(impl, Q1, torch.zeros_like(Q1))
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    invoke(impl, Q2, torch.zeros_like(Q2))
    torch.cuda.synchronize()
    assert time.perf_counter() - t0 < 2.0


# ================================================================ 独立运行入口

def run_standalone():
    assert HAS_CUDA, "需要 GPU"
    print(f"GPU: {torch.cuda.get_device_name(0)}，sm_{SM}")

    jobs = [("sync", torch.bfloat16, (2, 64)),
            ("sync", torch.bfloat16, (2, 100)),
            ("sync", torch.float32, (2, 100)),
            ("async", torch.bfloat16, (2, 64)),
            ("async", torch.bfloat16, (2, 100))]
    for name, dtype, (batch, N) in jobs:
        Q = torch.randn(batch, N, DK, device="cuda", dtype=dtype)
        Q_out = torch.zeros_like(Q)
        impl = IMPLS[name]()
        t0 = time.perf_counter()
        invoke(impl, Q, Q_out)
        torch.cuda.synchronize()
        dt = time.perf_counter() - t0
        ok = torch.equal(Q_out, Q)
        print(f"[{name}] B={batch} N={N} {dtype}: {'✓' if ok else '✗'} ({dt:.2f}s)")
        assert ok

    if SM < 90:
        print(f"[tma] SKIP（sm{SM} 不支持 TMA，接口已留好）")
    print("全部通过")


#if __name__ == "__main__":
run_standalone()

/tmp/ipykernel_461414/2752204273.py:24: DeprecationWarning: cutlass.utils.smem_allocator.SmemAllocator is deprecated; use cutlass.memory.SmemAllocator instead. It will be removed in a future release.
  smem = cutlass.utils.SmemAllocator()


GPU: NVIDIA GeForce RTX 5070 Ti，sm_120
[sync] B=2 N=64 torch.bfloat16: ✓ (0.01s)
[sync] B=2 N=100 torch.bfloat16: ✓ (0.01s)
[sync] B=2 N=100 torch.float32: ✓ (0.01s)
[async] B=2 N=64 torch.bfloat16: ✓ (0.06s)
[async] B=2 N=100 torch.bfloat16: ✓ (0.01s)
全部通过


## 2D copy

In [ ]:
"""tests/test_m3a2_copy2d.py — 3a 的第二步：二维 tile 的 cp.async 拷贝（行方向边界谓词 + 越界行补零）。

为什么要做这一步：
  flat 版把 (B,N,d) 当一维数组，B·N·64 恒为 8 的倍数，所以从没需要"行方向"的谓词。
  M3 里读 Q/K/V tile 时，N=100、Bq=64 的最后一个 tile 有 28 行越界，必须按行做谓词；
  而且 K/V 的越界行必须补零（否则 P=0 乘上 smem 里的垃圾，NaN×0=NaN，整个 O 被污染）。

设计：
  src (B, N, d) bf16；dst (B, Npad, d)，Npad = ceil(N/Bq)*Bq（比 src 多出的行用来检查补零）
  grid = (ceil(N/Bq), B)；每个 block 搬一个 (Bq, d) 的 tile：
      gmem --cp.async(行谓词)--> smem --(越界行补零)--> dst（整个 tile，不加谓词）
  线程布局 (32 行 × 8 列)，每线程 1 行 × 8 个 bf16 = 16B；Bq=64 → 沿行方向重复 2 次

分阶段上线（用 Python 静态开关，一次只加一个东西，哪步挂了就知道是哪块）：
  "nopred"  无谓词，只测 N 是 Bq 整数倍            —— 验证 tile 切分、smem layout、线程布局
  "pred"    加行谓词（越界行不读，smem 里是垃圾）     —— 验证坐标张量与谓词
  "full"    再加越界行补零                           —— 验证补零；这是 M3 要用的版本

【重要】带 [A?] 标记的 API 是我最没把握的点，报错时优先查它们：
  [A1] src3[(bidy, None, None)] 切 batch 维；备选 cute.slice_(src3, (bidy, None, None))
  [A2] 坐标张量 partition 之后的取值写法 tScS[(0, rv), m, k][0]（官方 SGEMM 例子里有类似写法）
  [A3] make_rmem_tensor 接受形状元组；备选 cute.make_fragment(layout, dtype)
  [A4] cutlass.BFloat16(0.0) 作为标量零；备选 tSsS.element_type(0.0)
  [A5] partition_S 的形状应为 ((8,1), CPY_M, CPY_K) = ((8,1), 2, 1)；用 debug=True 打印核对

跑法：
  uv run python tests/test_m3a2_copy2d.py          # 按阶段独立运行
  uv run pytest tests/test_m3a2_copy2d.py -v
  # 越界读检测（用精确分配，别让 torch 的缓存分配器吞掉小越界）：
  PYTORCH_NO_CUDA_MEMORY_CACHING=1 compute-sanitizer --tool memcheck uv run python tests/test_m3a2_copy2d.py
"""
import time

import pytest
import torch

import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack

D = 64                       # d 作为编译期常量：行 stride 静态 = D，编译器才能证明每行起点 16B 对齐
BQ = 64
HAS_CUDA = torch.cuda.is_available()


def to_cute(t: torch.Tensor):
    assert t.is_contiguous()
    return from_dlpack(t, assumed_align=16).mark_layout_dynamic(leading_dim=2)


def padded_rows(N: int) -> int:
    return (N + BQ - 1) // BQ * BQ


# ================================================================ 被测实现

class Copy2DAsync:
    def __init__(self, use_pred: bool = True, zero_fill: bool = True, debug: bool = False,
                 num_threads: int = 256):
        self.use_pred, self.zero_fill, self.debug = use_pred, zero_fill, debug
        self.NT = num_threads
        self.BQ, self.D, self.VEC = BQ, D, 8              # 8 个 bf16 = 16B = 128 位
        self.thr_n = D // self.VEC                        # 每行 8 个线程
        self.thr_m = num_threads // self.thr_n            # 32 行
        assert BQ % self.thr_m == 0, "Bq 必须是线程行数的整数倍"

    @cute.kernel
    def kernel(self, src3: cute.Tensor, dst3: cute.Tensor, tiled_copy: cute.TiledCopy):
        tidx, _, _ = cute.arch.thread_idx()
        bidx, bidy, _ = cute.arch.block_idx()             # bidx: 第几个 Q tile；bidy: 第几个 batch
        BQ_, D_, VEC = self.BQ, self.D, self.VEC

        smem = cutlass.utils.SmemAllocator()
        sTile = smem.allocate_tensor(src3.element_type, cute.make_layout((BQ_, D_), stride=(D_, 1)), 16)

        mSrc = src3[(bidy, None, None)]                   # [A1] (N, D)
        mDst = dst3[(bidy, None, None)]                   #      (Npad, D)
        N = mSrc.shape[0]                                 # 动态

        gSrc = cute.local_tile(mSrc, (BQ_, D_), (bidx, 0))          # (Bq, D)，最后一个 tile 可能越界
        gDst = cute.local_tile(mDst, (BQ_, D_), (bidx, 0))
        cSrc = cute.local_tile(cute.make_identity_tensor(mSrc.shape), (BQ_, D_), (bidx, 0))   # 坐标张量：元素是 (行, 列)

        thr = tiled_copy.get_slice(tidx)
        tSgS = thr.partition_S(gSrc)                      # 每线程 gmem 视图
        tSsS = thr.partition_D(sTile)                     # 每线程 smem 视图
        tScS = thr.partition_S(cSrc)                      # 每线程坐标视图（与 tSgS 同形状）
        tDgD = thr.partition_D(gDst)                      # 每线程输出视图（与 tSsS 同形状）

        if self.debug:                                    # Python 静态分支：trace 时打印 layout
            print("tSgS:", tSgS)                          # [A5] 预期 ((8,1),2,1)
            print("tSsS:", tSsS)
            print("tScS:", tScS)

        n_rep = tSgS.shape[0][1]                          # 一条指令内的向量重复数（静态，=1）
        n_m = cute.size(tSgS, mode=[1])                   # 沿行方向的重复数（静态，=Bq/32=2）
        n_k = cute.size(tSgS, mode=[2])                   # 沿列方向的重复数（静态，=1）

        # ---- gmem -> smem（cp.async）----
        if self.use_pred:
            # pred 是"每个 16B 向量一个"，形状 (n_rep, n_m, n_k)，不含向量内的 8
            pred = cute.make_rmem_tensor((n_rep, n_m, n_k), cutlass.Boolean)     # [A3]
            for rv in range(n_rep):                       # 普通 Python range：编译期展开
                for m in range(n_m):
                    for k in range(n_k):
                        row = tScS[(0, rv), m, k][0]      # [A2] 这个向量的行号（取向量内第 0 个元素的坐标）
                        pred[rv, m, k] = row < N
            cute.copy(tiled_copy, tSgS, tSsS, pred=pred)
        else:
            cute.copy(tiled_copy, tSgS, tSsS)

        cute.arch.cp_async_commit_group()
        cute.arch.cp_async_wait_group(0)
        cute.arch.barrier()                               # 本测试里每个线程只读自己搬的数据，其实不需要；
                                                          # 留着是为了养成习惯：真 kernel 里别的线程要读

        # ---- 越界行补零（K/V 的越界行必须是 0，否则 P·V 里 0×NaN = NaN）----
        if self.zero_fill:
            zero = cutlass.BFloat16(0.0)                  # [A4]
            for rv in range(n_rep):
                for m in range(n_m):
                    for k in range(n_k):
                        row = tScS[(0, rv), m, k][0]
                        if row >= N:
                            for e in range(VEC):          # VEC 是 Python int → 静态展开
                                tSsS[(e, rv), m, k] = zero

        # ---- smem -> dst：整个 tile 都写（含越界行，用来检查补零），不加谓词 ----
        for i in range(cute.size(tSsS)):                  # 线性下标，两个视图同形状，colex 顺序一致
            tDgD[i] = tSsS[i]

    @cute.jit
    def __call__(self, mSrc: cute.Tensor, mDst: cute.Tensor):
        assert mSrc.element_type == cutlass.BFloat16
        B = mSrc.shape[0]
        N = mSrc.shape[1]
        Np = mDst.shape[1]
        # 用 d 的静态常数重建 layout：行 stride = D（静态），这样编译器能证明每行起点 16B 对齐
        src3 = cute.make_tensor(mSrc.iterator, cute.make_layout((B, N, self.D), stride=(N * self.D, self.D, 1)))
        dst3 = cute.make_tensor(mDst.iterator, cute.make_layout((B, Np, self.D), stride=(Np * self.D, self.D, 1)))

        atom = cute.make_copy_atom(cute.nvgpu.cpasync.CopyG2SOp(), mSrc.element_type, num_bits_per_copy=128)
        tiled_copy = cute.make_tiled_copy_tv(
            atom,
            cute.make_layout((self.thr_m, self.thr_n), stride=(self.thr_n, 1)),   # 线程：32 行 × 8 列，相邻线程沿 d 连续 → 合并访问
            cute.make_layout((1, self.VEC)),                                      # 每线程：1 行 × 8 个元素
        )
        self.kernel(src3, dst3, tiled_copy).launch(
            grid=((N + self.BQ - 1) // self.BQ, B, 1),
            block=(self.NT, 1, 1),
            smem=self.BQ * self.D * 2,
        )


STAGES = {
    "nopred": dict(use_pred=False, zero_fill=False),
    "pred":   dict(use_pred=True,  zero_fill=False),
    "full":   dict(use_pred=True,  zero_fill=True),
}


def run(stage: str, B: int, N: int, seed: int = 0, sentinel: float = 123.0, debug: bool = False):
    """返回 (src, dst_padded)。dst 先填哨兵值：没被写到的行不会碰巧等于 0 或等于输入。"""
    torch.manual_seed(seed)
    Q = torch.randn(B, N, D, device="cuda", dtype=torch.bfloat16)
    out = torch.full((B, padded_rows(N), D), sentinel, device="cuda", dtype=torch.bfloat16)
    impl = Copy2DAsync(**STAGES[stage], debug=debug)
    impl(to_cute(Q), to_cute(out))
    torch.cuda.synchronize()
    return Q, out


def bits(t):
    return t.contiguous().view(torch.int16)


# ================================================================ 测试

needs_cuda = pytest.mark.skipif(not HAS_CUDA, reason="需要 GPU")


@needs_cuda
def test_smem_fits():
    optin = torch.cuda.get_device_properties(0).shared_memory_per_block_optin
    assert BQ * D * 2 <= optin, f"smem {BQ * D * 2} B 超过上限 {optin} B"


@needs_cuda
@pytest.mark.parametrize("stage", ["nopred", "pred", "full"])
@pytest.mark.parametrize("B,N", [(1, 64), (2, 128), (3, 192)])          # N 是 Bq 的整数倍：没有越界行
def test_aligned(stage, B, N):
    Q, out = run(stage, B, N)
    assert torch.equal(bits(out[:, :N]), bits(Q)), f"[{stage}] 整齐形状下 tile 切分/线程布局/smem layout 有问题"


@needs_cuda
@pytest.mark.parametrize("stage", ["pred", "full"])
@pytest.mark.parametrize("B,N", [(1, 100), (2, 100), (1, 129), (2, 63), (3, 1), (1, 200)])
def test_ragged_rows(stage, B, N):
    """不整除：最后一个 tile 部分越界。界内行必须逐位相等。"""
    Q, out = run(stage, B, N)
    assert torch.equal(bits(out[:, :N]), bits(Q)), f"[{stage}] B={B} N={N}：界内行不一致（谓词或坐标有误）"


@needs_cuda
@pytest.mark.parametrize("B,N", [(1, 100), (2, 63), (3, 1), (1, 129)])
def test_oob_rows_are_zero(B, N):
    """越界行必须是 0（哨值 123.0 已被覆盖）。K/V 的越界行不补零会让 P·V 变 NaN。"""
    _, out = run("full", B, N)
    tail = out[:, N:]
    assert tail.numel() > 0
    assert torch.equal(bits(tail), bits(torch.zeros_like(tail))), \
        f"越界行不是全 0：非零个数 {(tail != 0).sum().item()}，最大值 {tail.float().abs().max().item()}"


@needs_cuda
@pytest.mark.parametrize("trial", range(5))
def test_repeat_for_races(trial):
    """漏 wait / barrier 的 bug 是概率性的，多跑几遍。"""
    Q, out = run("full", 2, 100, seed=trial)
    assert torch.equal(bits(out[:, :100]), bits(Q)), f"第 {trial} 次不一致（竞态？）"


@needs_cuda
def test_bit_patterns():
    """NaN / ±Inf / ±0 的位模式原样通过：拷贝路径上不许有算术。"""
    torch.manual_seed(0)
    Q = torch.randn(1, 100, D, device="cuda", dtype=torch.bfloat16)
    Q[0, 0, 0], Q[0, 0, 1], Q[0, 0, 2], Q[0, 99, 3] = float("nan"), float("inf"), float("-inf"), float("-0.0")
    out = torch.full((1, padded_rows(100), D), 123.0, device="cuda", dtype=torch.bfloat16)
    Copy2DAsync(**STAGES["full"])(to_cute(Q), to_cute(out))
    torch.cuda.synchronize()
    assert torch.equal(bits(out[:, :100]), bits(Q))


@needs_cuda
def test_no_recompile_on_new_shape():
    impl = Copy2DAsync(**STAGES["full"])
    Q1 = torch.randn(1, 64, D, device="cuda", dtype=torch.bfloat16)
    o1 = torch.zeros(1, padded_rows(64), D, device="cuda", dtype=torch.bfloat16)
    impl(to_cute(Q1), to_cute(o1)); torch.cuda.synchronize()
    Q2 = torch.randn(2, 100, D, device="cuda", dtype=torch.bfloat16)
    o2 = torch.zeros(2, padded_rows(100), D, device="cuda", dtype=torch.bfloat16)
    t0 = time.perf_counter()
    impl(to_cute(Q2), to_cute(o2)); torch.cuda.synchronize()
    assert time.perf_counter() - t0 < 2.0, "换形状触发了重编译：d 应是静态、N 和 B 应是动态"


# ================================================================ 独立运行：按阶段逐步上线

def run_standalone():
    assert HAS_CUDA
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    # 第一次只打印 layout（debug=True），核对 [A5]
    t0 = time.perf_counter()
    run("nopred", 1, 64, debug=True)
    print(f"[编译 + 运行 nopred] {time.perf_counter() - t0:.2f}s")

    for stage, shapes in (("nopred", [(1, 64), (2, 128)]),
                          ("pred", [(1, 64), (2, 100), (1, 129), (3, 1)]),
                          ("full", [(2, 100), (1, 129), (3, 1)])):
        for B, N in shapes:
            Q, out = run(stage, B, N)
            ok_rows = torch.equal(bits(out[:, :N]), bits(Q))
            ok_zero = True
            if stage == "full" and out.shape[1] > N:
                ok_zero = bool((out[:, N:] == 0).all())
            print(f"[{stage:6s}] B={B} N={N:3d}: 界内 {'✓' if ok_rows else '✗'}"
                  + (f"  越界补零 {'✓' if ok_zero else '✗'}" if stage == "full" else ""))
            assert ok_rows and ok_zero
    print("全部通过")


if __name__ == "__main__":
    run_standalone()

/tmp/ipykernel_461414/3733513464.py:73: DeprecationWarning: cutlass.utils.smem_allocator.SmemAllocator is deprecated; use cutlass.memory.SmemAllocator instead. It will be removed in a future release.
  smem = cutlass.utils.SmemAllocator()


GPU: NVIDIA GeForce RTX 5070 Ti
tSgS: tensor<ptr<bf16, gmem, align<16>> o ((8,1),2,1):((1,0),2048,0)>
tSsS: tensor<ptr<bf16, smem, align<16>> o ((8,1),2,1):((1,0),2048,0)>
tScS: tensor<(?,?{div=8}) o ((8,1),2,1):((1@1,0),32@0,0)>
[编译 + 运行 nopred] 0.09s
tSgS: tensor<ptr<bf16, gmem, align<16>> o ((8,1),2,1):((1,0),2048,0)>
tSsS: tensor<ptr<bf16, smem, align<16>> o ((8,1),2,1):((1,0),2048,0)>
tScS: tensor<(?,?{div=8}) o ((8,1),2,1):((1@1,0),32@0,0)>
[nopred] B=1 N= 64: 界内 ✓
tSgS: tensor<ptr<bf16, gmem, align<16>> o ((8,1),2,1):((1,0),2048,0)>
tSsS: tensor<ptr<bf16, smem, align<16>> o ((8,1),2,1):((1,0),2048,0)>
tScS: tensor<(?,?{div=8}) o ((8,1),2,1):((1@1,0),32@0,0)>
[nopred] B=2 N=128: 界内 ✓
tSgS: tensor<ptr<bf16, gmem, align<16>> o ((8,1),2,1):((1,0),2048,0)>
tSsS: tensor<ptr<bf16, smem, align<16>> o ((8,1),2,1):((1,0),2048,0)>
tScS: tensor<(?,?{div=8}) o ((8,1),2,1):((1@1,0),32@0,0)>
[pred  ] B=1 N= 64: 界内 ✓
tSgS: tensor<ptr<bf16, gmem, align<16>> o ((8,1),2,1):((1,0),2048,0)>
tSsS: te